In [17]:
# -*- coding: utf-8 -*-
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: light
#       format_version: '1.5'
#       jupytext_version: 1.14.5
#   kernelspec:
#     display_name: Python 3 (ipykernel)
#     language: python
#     name: python3
# ---

# # Análise dos Resultados da Otimização de L0 via Algoritmos Genéticos

# ## 1. Configurações e Imports

# +
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import seaborn as sns
import os
import glob
from collections import Counter

# NOVAS IMPORTAÇÕES PARA CORRELAÇÃO E REGRESSÃO
from scipy.stats import pearsonr
import statsmodels.api as sm # Opcional para regressão mais detalhada

# Adicionar o diretório pai ao sys.path para importar o módulo da biblioteca
import sys
# Presumindo que o notebook está em 'examples', e a biblioteca em 'activetextclassification' no nível acima
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

try:
    from activetextclassification.visualization.ag_plots_evolution import plot_population_evolution_combined
    print("Função de plotagem 'plot_population_evolution_combined' importada.")
except ImportError as e:
    print(f"Erro ao importar 'plot_population_evolution_combined': {e}")
    print("Certifique-se que 'activetextclassification/visualization/ag_plots_evolution.py' existe e o PYTHONPATH está correto.")
    # Definir uma função placeholder para evitar que o resto do notebook quebre
    def plot_population_evolution_combined(*args, **kwargs):
        print("ERRO: plot_population_evolution_combined não pôde ser importada. Gráfico não gerado.")


# Configurações de Estilo para os Gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) 
plt.rcParams['axes.titlesize'] = 14 # Reduzido para subplots
plt.rcParams['axes.labelsize'] = 12 # Reduzido
plt.rcParams['xtick.labelsize'] = 10 # Reduzido
plt.rcParams['ytick.labelsize'] = 10 # Reduzido
plt.rcParams['legend.fontsize'] = 8  # Reduzido
plt.rcParams['figure.titlesize'] = 16 # Para suptitle

Função de plotagem 'plot_population_evolution_combined' importada.


In [18]:
# +
base_ag_results_path = "." 
l0_size_folders = glob.glob(os.path.join(base_ag_results_path, "ag_optimization_results_L0_*"))
AVAILABLE_L0_SIZES = sorted([int(os.path.basename(f).split('_')[-1]) for f in l0_size_folders if os.path.basename(f).split('_')[-1].isdigit()])
print(f"Tamanhos de L0 com pastas de resultados AG encontradas: {AVAILABLE_L0_SIZES}")

L0_SIZES_PRIMARY = [10, 50, 100, 500, 1000, 2500, 5000, 10000, 20000, 30000, 100000]
L0_SIZES_FOR_PLOTS = [s for s in L0_SIZES_PRIMARY if s in AVAILABLE_L0_SIZES]
L0_SIZES_FOR_CURVA_OTIMA = [s for s in L0_SIZES_PRIMARY if s in AVAILABLE_L0_SIZES and s <= 30000]
if not L0_SIZES_FOR_PLOTS and AVAILABLE_L0_SIZES: 
    L0_SIZES_FOR_PLOTS = AVAILABLE_L0_SIZES
elif not L0_SIZES_FOR_PLOTS and not AVAILABLE_L0_SIZES:
    print("ALERTA: Nenhum L0_SIZE disponível para os gráficos de 'Curva Ótima' e 'Características'.")
    L0_SIZES_FOR_PLOTS = [10] 

# Para as tabelas de correlação, é bom ter uma lista abrangente de L0s com dados
L0_FOR_CORR_TABLES = [s for s in L0_SIZES_PRIMARY if s in AVAILABLE_L0_SIZES]
if not L0_FOR_CORR_TABLES: L0_FOR_CORR_TABLES = AVAILABLE_L0_SIZES # Fallback

print(f"Tamanhos de L0 que serão considerados para as tabelas de correlação: {L0_FOR_CORR_TABLES}")

# Para os gráficos de evolução da população, usaremos os exemplos da orientação
# ou o que estiver disponível.
CONVERGENCE_L0_SIZES_EXAMPLE = [s for s in [10, 50, 100, 500, 1000, 2500, 5000, 10000, 20000, 30000] if s in AVAILABLE_L0_SIZES]
if not CONVERGENCE_L0_SIZES_EXAMPLE and AVAILABLE_L0_SIZES: # Se os da orientação não existem, pega os primeiros disponíveis
    CONVERGENCE_L0_SIZES_EXAMPLE = AVAILABLE_L0_SIZES[:min(4, len(AVAILABLE_L0_SIZES))]
elif not CONVERGENCE_L0_SIZES_EXAMPLE and not AVAILABLE_L0_SIZES: # Se nada disponível
     print("ALERTA: Nenhum L0_SIZE disponível para os gráficos de 'Evolução da População'.")
     CONVERGENCE_L0_SIZES_EXAMPLE = []


print(f"Tamanhos de L0 para gráficos de 'Curva Ótima' e 'Características': {L0_SIZES_FOR_PLOTS}")
print(f"Tamanhos de L0 para gráficos de 'Evolução da População': {CONVERGENCE_L0_SIZES_EXAMPLE}")

RANDOM_STATS_FILE = os.path.join("data", "sensibilidade", "estatísticas.csv") 
FULL_DATASET_FILE = os.path.join("..", "data", "dataset.csv") 
TEXT_COLUMN = 'nm_item'; LABEL_COLUMN = 'nm_product'
AG_BEST_L0_BASE_NAME = "ag_best_l0"; AG_DETAILED_FITNESS_BASE_NAME = "ag_detailed_fitness" 
METRIC_MAP = {"ACCURACY": "Acurácia", "F1": "Macro F1-Score"}
GOAL_MAP = {"MAXIMIZE": "Maximização", "MINIMIZE": "Minimização"}

MAX_GENERATIONS_TO_PLOT = 100 # Limite de gerações para os plots
L0_SIZE_LIMIT_CURVA_OTIMA = 30000
MARKER_SIZE_CURVA_OTIMA = 4 # Novo parâmetro para tamanho dos marcadores na Seção 6

Tamanhos de L0 com pastas de resultados AG encontradas: [10, 50, 100, 500, 1000, 2500, 5000, 10000, 20000, 30000]
Tamanhos de L0 que serão considerados para as tabelas de correlação: [10, 50, 100, 500, 1000, 2500, 5000, 10000, 20000, 30000]
Tamanhos de L0 para gráficos de 'Curva Ótima' e 'Características': [10, 50, 100, 500, 1000, 2500, 5000, 10000, 20000, 30000]
Tamanhos de L0 para gráficos de 'Evolução da População': [10, 50, 100, 500, 1000, 2500, 5000, 10000, 20000, 30000]


In [19]:
# ## 3. Carregamento de Dados 
# (Mantido como na versão anterior)

# ### 3.1 Dados da Amostragem Aleatória (do `estatisticas.csv`)

# +
df_random_detailed_stats = None
try:
    df_random_detailed_stats = pd.read_csv(RANDOM_STATS_FILE)
    print(f"Dados detalhados de amostragem aleatória carregados de: {RANDOM_STATS_FILE}")
    # Renomear colunas para consistência, se necessário (parecem ok pela amostra)
    # Converter l0_size para int se não for
    if 'l0_size' in df_random_detailed_stats.columns:
        df_random_detailed_stats['l0_size'] = pd.to_numeric(df_random_detailed_stats['l0_size'], errors='coerce')
        df_random_detailed_stats.dropna(subset=['l0_size'], inplace=True)
    else:
        print(f"AVISO: Coluna 'l0_size' não encontrada em {RANDOM_STATS_FILE}")
except FileNotFoundError: 
    print(f"ERRO: Arquivo de estatísticas aleatórias '{RANDOM_STATS_FILE}' não encontrado.")
except Exception as e: 
    print(f"ERRO ao carregar dados da amostragem aleatória de '{RANDOM_STATS_FILE}': {e}")

if df_random_detailed_stats is not None:
    display(df_random_detailed_stats.head())
# -

# ### 3.2 Dataset Completo (para análise de características dos L0s)

# +
df_full = None
try:
    df_full = pd.read_csv(FULL_DATASET_FILE)
    df_full.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN], inplace=True)
except FileNotFoundError: print(f"ERRO: Dataset completo não encontrado: {FULL_DATASET_FILE}"); df_full = None 
except Exception as e: print(f"ERRO ao carregar o dataset completo: {e}"); df_full = None
# -

Dados detalhados de amostragem aleatória carregados de: data\sensibilidade\estatísticas.csv


,Métrica,l0_size,Média,Mediana,DesvioPadrão,Mínimo,P25,P75,Máximo,IQR,CV (Mediana)
0,Acurácia,10,0.066649,0.068869,0.023793,0.025603,0.045918,0.088246,0.102171,0.042328,0.345485
1,Acurácia,20,0.105012,0.110059,0.023253,0.049388,0.089185,0.121807,0.138118,0.032622,0.211274
2,Acurácia,30,0.139531,0.138678,0.022874,0.095281,0.125312,0.155383,0.181395,0.030071,0.164945
3,Acurácia,40,0.159713,0.159447,0.022555,0.100274,0.144589,0.178245,0.191021,0.033656,0.141460
4,Acurácia,50,0.178486,0.179758,0.021830,0.124079,0.164435,0.198550,0.208496,0.034115,0.121440


In [30]:
# ## 4. Funções Auxiliares
# (Função load_and_aggregate_detailed_log CORRIGIDA)

# +
def get_ag_best_l0_filepath(l0_size, metric_short, goal_short):
    dir_path = os.path.join(base_ag_results_path, f"ag_optimization_results_L0_{l0_size}")
    filename = f"{AG_BEST_L0_BASE_NAME}_{metric_short.upper()}_{goal_short.upper()}.csv"
    return os.path.join(dir_path, filename)

def get_ag_detailed_fitness_filepath(l0_size, metric_short, goal_short):
    dir_path = os.path.join(base_ag_results_path, f"ag_optimization_results_L0_{l0_size}")
    filename = f"{AG_DETAILED_FITNESS_BASE_NAME}{metric_short.upper()}_{goal_short.upper()}.csv"
    return os.path.join(dir_path, filename)

# Função load_and_aggregate_detailed_log CORRIGIDA (essencial)
def load_and_aggregate_detailed_log(l0_size, metric_short_name, goal_short_name, max_gens=None):
    filepath = get_ag_detailed_fitness_filepath(l0_size, metric_short_name, goal_short_name)
    metric_col_in_file = 'accuracy_on_full' if metric_short_name.upper() == 'ACCURACY' else 'f1_macro_on_full'
    try:
        df_detailed = pd.read_csv(filepath)
        if max_gens and 'generation' in df_detailed.columns:
            df_detailed = df_detailed[df_detailed['generation'] <= max_gens]
        if df_detailed.empty: return None
        if metric_col_in_file not in df_detailed.columns: return None
        df_detailed[metric_col_in_file] = pd.to_numeric(df_detailed[metric_col_in_file], errors='coerce')
        char_cols = ['num_tokens', 'num_distinct_tokens', 'num_classes_in_l0']
        for char_col in char_cols:
            if char_col in df_detailed.columns:
                df_detailed[char_col] = pd.to_numeric(df_detailed[char_col], errors='coerce')
            else: df_detailed[char_col] = np.nan
        df_detailed.dropna(subset=[metric_col_in_file], inplace=True) 
        if df_detailed.empty: return None
        grouped = df_detailed.groupby('generation')
        df_agg_perf = grouped[metric_col_in_file].agg(['max', 'mean', 'min']).reset_index()
        df_agg_perf.rename(columns={'max': 'max_metric', 'mean': 'avg_metric', 'min': 'min_metric'}, inplace=True)
        idx_best_worst = grouped[metric_col_in_file].idxmax() if goal_short_name.upper() == 'MAXIMIZE' else grouped[metric_col_in_file].idxmin()
        df_best_worst_chars_raw = df_detailed.loc[idx_best_worst]
        existing_char_cols_for_merge = ['generation'] + [col for col in char_cols if col in df_best_worst_chars_raw.columns]
        df_best_worst_chars = df_best_worst_chars_raw[existing_char_cols_for_merge].reset_index(drop=True)
        if df_best_worst_chars.empty or not any(col in char_cols for col in df_best_worst_chars.columns):
            df_agg_final = df_agg_perf.copy()
            for col in char_cols: df_agg_final[col] = np.nan
        else:
            df_agg_final = pd.merge(df_agg_perf, df_best_worst_chars, on='generation', how='left')
        for col in char_cols:
            if col not in df_agg_final.columns: df_agg_final[col] = np.nan
        return df_agg_final
    except FileNotFoundError: return None
    except Exception as e:
        print(f"Erro em load_and_aggregate_detailed_log para L0={l0_size}, Métrica={metric_short_name}, Obj={goal_short_name}: {type(e).__name__} - {e}")
        return None

def load_detailed_log_raw(l0_size, metric_short_name, goal_short_name, max_gens=None):
    filepath = get_ag_detailed_fitness_filepath(l0_size, metric_short_name, goal_short_name)
    metric_col_in_file = 'accuracy_on_full' if metric_short_name.upper() == 'ACCURACY' else 'f1_macro_on_full'
    char_cols_from_log = ['num_tokens', 'num_distinct_tokens', 'num_classes_in_l0']
    try:
        df_detailed = pd.read_csv(filepath)
        if max_gens and 'generation' in df_detailed.columns:
            df_detailed = df_detailed[df_detailed['generation'] <= max_gens]
        if df_detailed.empty: return None
        required_cols = [metric_col_in_file, 'generation'] + char_cols_from_log
        if not all(col in df_detailed.columns for col in required_cols): return None
        df_detailed[metric_col_in_file] = pd.to_numeric(df_detailed[metric_col_in_file], errors='coerce')
        for char_col in char_cols_from_log:
            if char_col in df_detailed.columns:
                df_detailed[char_col] = pd.to_numeric(df_detailed[char_col], errors='coerce')
            else: df_detailed[char_col] = np.nan # Adicionar coluna se não existir
        # Apenas dropna se a métrica de performance ou características essenciais para correlação forem NaN
        # Para a correlação, precisamos de pares válidos.
        df_detailed.dropna(subset=[metric_col_in_file] + char_cols_from_log + ['generation'], how='any', inplace=True) 
        if df_detailed.empty: return None
        return df_detailed[['generation', metric_col_in_file] + char_cols_from_log]
    except FileNotFoundError: return None
    except Exception as e:
        print(f"Erro em load_detailed_log_raw para L0={l0_size}, Métrica={metric_short_name}, Obj={goal_short_name}: {type(e).__name__} - {e}")
        return None
    
# --- NOVA FUNÇÃO ---
def load_all_individual_data(l0_sizes_to_load, max_gens_to_load):
    """
    Carrega e concatena todos os dados de indivíduos dos logs detalhados
    para os L0 sizes, métricas e objetivos especificados.
    """
    master_df_list = []
    print(f"\n--- Carregando dados de todos os indivíduos para L0s: {l0_sizes_to_load} (até {max_gens_to_load} ger.) ---")

    for l0_size in l0_sizes_to_load:
        for metric_short in ["ACCURACY", "F1"]:
            for goal_short in ["MAXIMIZE", "MINIMIZE"]:
                # print(f"  Carregando: L0={l0_size}, Métrica={metric_short}, Objetivo={goal_short}")
                df_segment = load_detailed_log_raw(l0_size, metric_short, goal_short, max_gens=max_gens_to_load)
                if df_segment is not None and not df_segment.empty:
                    df_segment['L0 Size'] = l0_size
                    df_segment['Métrica Base'] = METRIC_MAP[metric_short] # Acurácia ou Macro F1-Score
                    df_segment['Objetivo Original'] = GOAL_MAP[goal_short] # Maximização ou Minimização
                    master_df_list.append(df_segment)
    
    if not master_df_list:
        print("Nenhum dado de indivíduo foi carregado.")
        return pd.DataFrame() # Retorna DataFrame vazio
        
    df_all_individuals = pd.concat(master_df_list, ignore_index=True)
    print(f"Total de observações de indivíduos carregadas: {len(df_all_individuals)}")
    
    # Limpeza final: converter para tipos corretos e lidar com possíveis NaNs introduzidos
    # As colunas de performance e características já devem ter sido convertidas em load_detailed_log_raw
    # Mas uma verificação não faz mal.
    cols_to_check_numeric = ['performance_accuracy', 'performance_f1', 'num_tokens', 'num_distinct_tokens', 'num_classes_in_l0', 'generation', 'L0 Size']
    for col in cols_to_check_numeric:
        if col in df_all_individuals.columns:
            df_all_individuals[col] = pd.to_numeric(df_all_individuals[col], errors='coerce')
            
    # Remover linhas onde colunas essenciais para qualquer análise de correlação são NaN
    # Essencialmente, precisamos de pelo menos uma performance e uma característica.
    # Mas para simplificar, vamos garantir que todas as colunas de interesse tenham valores não-NaN para as correlações.
    # No entanto, o dropna deve ser feito no momento do cálculo da correlação para cada par específico.
    # Apenas dropar se a geração ou L0 size for NaN.
    df_all_individuals.dropna(subset=['generation', 'L0 Size'], inplace=True)

    return df_all_individuals
# -
# -
# ## Chamada da Nova Função para Carregar Todos os Dados
# (Esta célula deve ser executada uma vez para popular `df_master_all_individuals`)

# +
# Usar L0_FOR_CORR_TABLES ou AVAILABLE_L0_SIZES para carregar dados para todos os L0s relevantes
df_master_all_individuals = load_all_individual_data(L0_FOR_CORR_TABLES, MAX_GENERATIONS_TO_PLOT)

if df_master_all_individuals.empty:
    print("ALERTA: Nenhum dado de indivíduo foi carregado no DataFrame mestre. As análises de correlação podem não funcionar.")
else:
    print("\nPrimeiras linhas do DataFrame Mestre de Indivíduos:")
    display(df_master_all_individuals.head())
    print("\nTipos de dados do DataFrame Mestre:")
    display(df_master_all_individuals.info())
# -


--- Carregando dados de todos os indivíduos para L0s: [10, 50, 100, 500, 1000, 2500, 5000, 10000, 20000, 30000] (até 100 ger.) ---
Total de observações de indivíduos carregadas: 80000

Primeiras linhas do DataFrame Mestre de Indivíduos:


,generation,accuracy_on_full,num_tokens,num_distinct_tokens,num_classes_in_l0,L0 Size,Métrica Base,Objetivo Original,f1_macro_on_full
0,1,0.089022,49,47,9,10,Acurácia,Maximização,NaN
1,1,0.032768,58,54,9,10,Acurácia,Maximização,NaN
2,1,0.025199,58,58,8,10,Acurácia,Maximização,NaN
3,1,0.041092,54,52,9,10,Acurácia,Maximização,NaN
4,1,0.061770,57,56,9,10,Acurácia,Maximização,NaN



Tipos de dados do DataFrame Mestre:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   generation           80000 non-null  int64  
 1   accuracy_on_full     40000 non-null  float64
 2   num_tokens           80000 non-null  int64  
 3   num_distinct_tokens  80000 non-null  int64  
 4   num_classes_in_l0    80000 non-null  int64  
 5   L0 Size              80000 non-null  int64  
 6   Métrica Base         80000 non-null  object 
 7   Objetivo Original    80000 non-null  object 
 8   f1_macro_on_full     40000 non-null  float64
dtypes: float64(2), int64(5), object(2)
memory usage: 5.5+ MB


None

In [33]:
df_master_all_individuals.to_excel("allag.xlsx", index=False)

In [21]:
# ## 9. (Renumerado) Cálculo de `df_corr_summary` (Correlação por Cenário Individual)
# (Esta é a antiga Seção "Novo Código" ou "Seção 11" que gerava df_corr_summary)

# +
correlation_results_individual_scenario = [] 
characteristic_cols = ['num_tokens', 'num_distinct_tokens', 'num_classes_in_l0'] 
performance_metrics_cols_dict = { 
    "ACCURACY": "accuracy_on_full",
    "F1": "f1_macro_on_full"
}
scenarios_for_individual_corr = [
    ("ACCURACY", "MAXIMIZE"), ("ACCURACY", "MINIMIZE"),
    ("F1", "MAXIMIZE"), ("F1", "MINIMIZE")
]

l0_sizes_for_individual_corr = L0_FOR_CORR_TABLES 

df_corr_summary = pd.DataFrame() 

if not l0_sizes_for_individual_corr:
    print("Nenhum L0_SIZE definido em L0_FOR_CORR_TABLES. Pulando cálculo de df_corr_summary.")
else:
    print(f"\n--- Calculando Correlações por Cenário Individual para L0s: {l0_sizes_for_individual_corr} (até {MAX_GENERATIONS_TO_PLOT} ger.) ---")
    for l0_size_corr_ind in l0_sizes_for_individual_corr:
        for metric_short_corr_ind, goal_short_corr_ind in scenarios_for_individual_corr:
            df_raw_ind = load_detailed_log_raw(l0_size_corr_ind, metric_short_corr_ind, goal_short_corr_ind, max_gens=MAX_GENERATIONS_TO_PLOT)
            
            if df_raw_ind is None or df_raw_ind.empty:
                # print(f"  L0={l0_size_corr_ind}, {metric_short_corr_ind}-{goal_short_corr_ind}: Sem dados RAW.")
                # Adicionar entradas NaN para manter a estrutura da tabela
                for char_col_corr_ind_nan in characteristic_cols:
                    correlation_results_individual_scenario.append({
                        "L0 Size": l0_size_corr_ind, "Métrica Otimizada": METRIC_MAP[metric_short_corr_ind],
                        "Objetivo": GOAL_MAP[goal_short_corr_ind], "Característica": char_col_corr_ind_nan,
                        "Correlação Pearson (r)": np.nan, "p-valor (correlação)": np.nan,
                        "R² (Regressão Linear)": np.nan, "p-valor (slope)": np.nan,
                        "N Observações": 0
                    })
                continue # Pula para o próximo cenário
            
            perf_col_name_ind = performance_metrics_cols_dict[metric_short_corr_ind]
            
            for char_col_corr_ind in characteristic_cols:
                # Verificar novamente se as colunas existem após o carregamento (load_detailed_log_raw já verifica)
                if char_col_corr_ind not in df_raw_ind.columns or perf_col_name_ind not in df_raw_ind.columns:
                     correlation_results_individual_scenario.append({
                        "L0 Size": l0_size_corr_ind, "Métrica Otimizada": METRIC_MAP[metric_short_corr_ind],
                        "Objetivo": GOAL_MAP[goal_short_corr_ind], "Característica": char_col_corr_ind,
                        "Correlação Pearson (r)": np.nan, "p-valor (correlação)": np.nan,
                        "R² (Regressão Linear)": np.nan, "p-valor (slope)": np.nan,
                        "N Observações": 0
                    })
                     continue

                df_pair_ind = df_raw_ind[[perf_col_name_ind, char_col_corr_ind]].dropna()
                r_ind, p_corr_ind, r_sq_ind, p_slope_ind = np.nan, np.nan, np.nan, np.nan
                n_obs = len(df_pair_ind)

                if n_obs >= 2: 
                    try:
                        if n_obs > 2: r_ind, p_corr_ind = pearsonr(df_pair_ind[perf_col_name_ind], df_pair_ind[char_col_corr_ind])
                        elif n_obs == 2: # r é calculável, p-valor não
                             r_ind_calc = np.corrcoef(df_pair_ind[perf_col_name_ind], df_pair_ind[char_col_corr_ind])[0,1]
                             if np.isclose(abs(r_ind_calc), 1.0): r_ind = r_ind_calc
                             else: r_ind = np.nan # Evitar r indefinido se não for perfeitamente correlacionado com N=2
                             p_corr_ind = np.nan
                    except (ValueError, IndexError): r_ind, p_corr_ind = np.nan, np.nan
                
                if n_obs >=3: # Para OLS
                    X_ind = df_pair_ind[char_col_corr_ind]
                    y_ind = df_pair_ind[perf_col_name_ind]
                    if X_ind.nunique() > 1: 
                        X_ind_sm = sm.add_constant(X_ind, has_constant='add')
                        try:
                            model_fit_ind = sm.OLS(y_ind, X_ind_sm).fit()
                            r_sq_ind = model_fit_ind.rsquared
                            if len(model_fit_ind.pvalues) > 1: p_slope_ind = model_fit_ind.pvalues.iloc[1]
                        except Exception: pass 
                        
                correlation_results_individual_scenario.append({
                    "L0 Size": l0_size_corr_ind, "Métrica Otimizada": METRIC_MAP[metric_short_corr_ind],
                    "Objetivo": GOAL_MAP[goal_short_corr_ind], "Característica": char_col_corr_ind,
                    "Correlação Pearson (r)": r_ind, "p-valor (correlação)": p_corr_ind,
                    "R² (Regressão Linear)": r_sq_ind, "p-valor (slope)": p_slope_ind,
                    "N Observações": n_obs
                })

    if correlation_results_individual_scenario:
        df_corr_summary = pd.DataFrame(correlation_results_individual_scenario)
        print("\n--- Tabela Resumo de Correlações Individuais por Cenário (df_corr_summary) Gerada ---")
    else:
        print("Nenhum resultado de correlação individual para df_corr_summary.")
        df_corr_summary = pd.DataFrame(columns=[ # Criar com colunas esperadas se vazio
            "L0 Size", "Métrica Otimizada", "Objetivo", "Característica",
            "Correlação Pearson (r)", "p-valor (correlação)",
            "R² (Regressão Linear)", "p-valor (slope)", "N Observações"
        ])
# -


--- Calculando Correlações por Cenário Individual para L0s: [10, 50, 100, 500, 1000, 2500, 5000, 10000, 20000, 30000] (até 100 ger.) ---

--- Tabela Resumo de Correlações Individuais por Cenário (df_corr_summary) Gerada ---


In [25]:
# ## 10. (Renumerado) Tabelas de Interpretação de Correlação (Heatmaps Textuais - Usa `df_corr_summary`)
# (Esta era a Seção 15 na sua solicitação, agora 10, dependendo da Seção 9 acima)

# +
# (Código da Seção 15 (Heatmaps) da sua última resposta, que usa df_corr_summary, COLADO AQUI)
# Limiares e descrições para correlação
CORR_THRESHOLDS = {
    "Forte Positiva": (0.7, 1.01), "Moderada Positiva": (0.4, 0.7), "Fraca Positiva": (0.1, 0.4),
    "Insensível": (-0.1, 0.1), "Fraca Negativa": (-0.4, -0.1),
    "Moderada Negativa": (-0.7, -0.4), "Forte Negativa": (-1.01, -0.7),
}
P_VALUE_SIGNIFICANCE_HEATMAP = 0.05 # Renomeado para evitar conflito se P_VALUE_SIGNIFICANCE for usado em outro lugar

CORR_COLORS_BG = {
    "Forte Positiva":    '#d62728', "Moderada Positiva": '#ff7f0e', "Fraca Positiva":    '#ffbb78',
    "Insensível":        '#f0f0f0', "Fraca Negativa":    '#aec7e8', "Moderada Negativa": '#1f77b4',
    "Forte Negativa":    '#084594', "Não Significativa": '#e0e0e0' 
}
CORR_COLORS_TEXT = {
    "Forte Positiva":    'white', "Moderada Positiva": 'black', "Fraca Positiva":    'black',
    "Insensível":        'black', "Fraca Negativa":    'black', "Moderada Negativa": 'white',
    "Forte Negativa":    'white', "Não Significativa": 'dimgray'
}

def interpret_correlation_for_heatmap(r, p_value): # Renomeado para clareza
    if pd.isna(r): # p_value pode ser NaN se r for NaN ou N<3
        return "N/A", "N/A", "lightgrey", "black" 
    
    text_val_display = f"{r:.2f}" # O que será exibido na célula se não for a descrição

    if pd.notna(p_value) and p_value >= P_VALUE_SIGNIFICANCE_HEATMAP:
        desc = f"Insensível (p={p_value:.2g})"
        bg_color = CORR_COLORS_BG["Não Significativa"]; text_color = CORR_COLORS_TEXT["Não Significativa"]
        # Para heatmap só com cores, retornaremos apenas r e as cores
        return r, desc, bg_color, text_color # Modificado para retornar r original para o styler.format

    # Se significativo
    for desc_key, (low, high) in CORR_THRESHOLDS.items():
        if low <= r < high:
            # desc_with_r_val = f"{desc_key} (r={r:.2f})" # Não precisamos mais disso para display
            if desc_key == "Insensível": # Já é significativo, então r está próximo de zero
                 desc_for_display = f"Próx. Zero (r={r:.2f}, p={p_value:.2g})"
                 bg_color = CORR_COLORS_BG["Insensível"]; text_color = CORR_COLORS_TEXT["Insensível"]
            else:
                desc_for_display = f"{desc_key} (r={r:.2f})" # Manter este para referência, mas não será o display principal
                bg_color = CORR_COLORS_BG[desc_key]; text_color = CORR_COLORS_TEXT[desc_key]
            return r, desc_for_display, bg_color, text_color # Retorna r original, descrição completa, e cores
    return r, "Erro", "white", "black" # Fallback

def style_cell_for_heatmap(val_tuple_r_desc_colors): 
    if isinstance(val_tuple_r_desc_colors, tuple) and len(val_tuple_r_desc_colors) == 4:
        _, _, bg_color, text_color = val_tuple_r_desc_colors # Ignora r e desc para o estilo direto
        return f'background-color: {bg_color}; color: {text_color};'
    return '' # Estilo vazio se não for tupla


if 'df_corr_summary' not in locals() or df_corr_summary is None or df_corr_summary.empty:
    print("AVISO: df_corr_summary não está disponível ou está vazio. Pulando Seção 10 (Heatmaps).")
else:
    char_display_map_heatmap = { # Renomeado para evitar conflito
        'num_tokens': 'Nº Total de Tokens',
        'num_distinct_tokens': 'Nº Tokens Distintos',
        'num_classes_in_l0': 'Nº Classes Únicas no L0'
    }
    for char_col_table, char_display_table in char_display_map_heatmap.items():
        print(f"\n\n--- Heatmap de Correlação para: {char_display_table} ---")
        df_char_corr_heatmap_src = df_corr_summary[df_corr_summary['Característica'] == char_col_table].copy()
        if df_char_corr_heatmap_src.empty: print(f"  Sem dados para {char_display_table}."); continue

        # Coluna para aplicar o Styler (contém tuplas com r, p_val, e info de estilo)
        df_char_corr_heatmap_src['StyleInfo'] = df_char_corr_heatmap_src.apply(
            lambda row: interpret_correlation_for_heatmap(row['Correlação Pearson (r)'], row['p-valor (correlação)']), axis=1
        )
        
        try:
            # Tabela pivotada com os valores de 'r' para exibição
            pivot_table_display_r = df_char_corr_heatmap_src.pivot_table(
                index="L0 Size", columns=["Métrica Otimizada", "Objetivo"],
                values="Correlação Pearson (r)", aggfunc='first')

            # Tabela pivotada com as tuplas de 'StyleInfo' para aplicar estilos
            pivot_table_for_style = df_char_corr_heatmap_src.pivot_table(
                index="L0 Size", columns=["Métrica Otimizada", "Objetivo"],
                values="StyleInfo", aggfunc='first')
                
        except Exception as e: print(f"Erro ao pivotar para heatmap de {char_display_table}: {e}"); continue
            
        if pivot_table_display_r.empty: print(f"  Tabela pivotada (r) vazia para {char_display_table}."); continue

        # Função para aplicar o estilo dinamicamente
        def _apply_heatmap_styles(data_series_of_r_values): # data_series_of_r_values é uma coluna da pivot_table_display_r
            # O nome da série (data_series_of_r_values.name) é uma tupla (Métrica Otimizada, Objetivo)
            # O índice da série (data_series_of_r_values.index) é o L0 Size
            
            # Recuperar as tuplas de estilo da pivot_table_for_style
            # E aplicar estilo baseado nelas
            
            styles = []
            for l0_idx, r_val in data_series_of_r_values.items():
                # Acessar a tupla de estilo correspondente
                metric_col_level0, obj_col_level1 = data_series_of_r_values.name
                style_tuple = pivot_table_for_style.loc[l0_idx, (metric_col_level0, obj_col_level1)]
                styles.append(style_cell_for_heatmap(style_tuple))
            return styles

        styled_heatmap = pivot_table_display_r.style.format("{:.2f}", na_rep="N/A")\
            .apply(_apply_heatmap_styles, axis=0)\
            .set_table_styles([
                {'selector': 'th', 'props': [('font-size', '8pt'), ('text-align', 'center'), ('padding', '3px')]}, # Reduzido th
                {'selector': 'td', 'props': [('font-size', '8pt'), ('text-align', 'center'), ('min-width', '70px'), ('padding', '3px')]}  # Reduzido td
            ]).set_caption(f"Heatmap Correlação (r): {char_display_table} vs Performance (Corpo: r, Fundo: Força/Signif.)")

        try:
            from IPython.display import display, HTML; display(styled_heatmap)
        except ImportError: print(pivot_table_display_r.to_string())
# -



--- Heatmap de Correlação para: Nº Total de Tokens ---




--- Heatmap de Correlação para: Nº Tokens Distintos ---




--- Heatmap de Correlação para: Nº Classes Únicas no L0 ---


In [26]:
# ## 4. Funções Auxiliares
# (Essas funções são necessárias e devem incluir a versão corrigida de load_and_aggregate_detailed_log)

# +
def get_ag_detailed_fitness_filepath(l0_size, metric_short, goal_short):
    dir_path = os.path.join(base_ag_results_path, f"ag_optimization_results_L0_{l0_size}")
    filename = f"ag_detailed_fitness{metric_short.upper()}_{goal_short.upper()}.csv"
    return os.path.join(dir_path, filename)

def load_and_aggregate_detailed_log(l0_size, metric_short_name, goal_short_name, max_gens=None):
    filepath = get_ag_detailed_fitness_filepath(l0_size, metric_short_name, goal_short_name)
    metric_col_in_file = 'accuracy_on_full' if metric_short_name.upper() == 'ACCURACY' else 'f1_macro_on_full'
    try:
        df_detailed = pd.read_csv(filepath)
        if max_gens and 'generation' in df_detailed.columns:
            df_detailed = df_detailed[df_detailed['generation'] <= max_gens]
        if df_detailed.empty: return None
        if metric_col_in_file not in df_detailed.columns: return None
        df_detailed[metric_col_in_file] = pd.to_numeric(df_detailed[metric_col_in_file], errors='coerce')
        char_cols = ['num_tokens', 'num_distinct_tokens', 'num_classes_in_l0']
        for char_col in char_cols:
            if char_col in df_detailed.columns:
                df_detailed[char_col] = pd.to_numeric(df_detailed[char_col], errors='coerce')
            else: df_detailed[char_col] = np.nan
        df_detailed.dropna(subset=[metric_col_in_file], inplace=True) 
        if df_detailed.empty: return None
        grouped = df_detailed.groupby('generation')
        df_agg_perf = grouped[metric_col_in_file].agg(['max', 'mean', 'min']).reset_index()
        df_agg_perf.rename(columns={'max': 'max_metric', 'mean': 'avg_metric', 'min': 'min_metric'}, inplace=True)
        
        if goal_short_name.upper() == 'MAXIMIZE':
            idx_best_worst = grouped[metric_col_in_file].idxmax()
        else: 
            idx_best_worst = grouped[metric_col_in_file].idxmin()
        
        df_best_worst_chars_raw = df_detailed.loc[idx_best_worst]
        existing_char_cols_for_merge = ['generation'] + [col for col in char_cols if col in df_best_worst_chars_raw.columns]
        df_best_worst_chars = df_best_worst_chars_raw[existing_char_cols_for_merge].reset_index(drop=True)

        if df_best_worst_chars.empty or not any(col in char_cols for col in df_best_worst_chars.columns if col in df_best_worst_chars): # Check if any relevant char_col exists
            df_agg_final = df_agg_perf.copy()
            for col in char_cols: df_agg_final[col] = np.nan
        else:
            df_agg_final = pd.merge(df_agg_perf, df_best_worst_chars, on='generation', how='left')
        
        for col in char_cols: # Ensure all characteristic columns are present, even if all NaN
            if col not in df_agg_final.columns:
                df_agg_final[col] = np.nan
        return df_agg_final
    except FileNotFoundError: return None
    except Exception as e:
        print(f"Erro em load_and_aggregate_detailed_log para L0={l0_size}, Métrica={metric_short_name}, Obj={goal_short_name}: {type(e).__name__} - {e}")
        return None

def load_detailed_log_raw(l0_size, metric_short_name, goal_short_name, max_gens=None):
    filepath = get_ag_detailed_fitness_filepath(l0_size, metric_short_name, goal_short_name)
    metric_col_in_file = 'accuracy_on_full' if metric_short_name.upper() == 'ACCURACY' else 'f1_macro_on_full'
    char_cols_from_log = ['num_tokens', 'num_distinct_tokens', 'num_classes_in_l0']
    try:
        df_detailed = pd.read_csv(filepath)
        if max_gens and 'generation' in df_detailed.columns:
            df_detailed = df_detailed[df_detailed['generation'] <= max_gens]
        if df_detailed.empty: return None
        
        # Checagem mais robusta de colunas
        required_cols_check = [metric_col_in_file, 'generation'] + char_cols_from_log
        for col_check in required_cols_check:
            if col_check not in df_detailed.columns:
                # print(f"AVISO: Coluna RAW '{col_check}' ausente em {filepath} para L0={l0_size}.")
                # Adicionar coluna com NaNs se não existir, para evitar falhas posteriores
                if col_check not in df_detailed.columns: df_detailed[col_check] = np.nan

        df_detailed[metric_col_in_file] = pd.to_numeric(df_detailed[metric_col_in_file], errors='coerce')
        for char_col in char_cols_from_log:
             if char_col in df_detailed.columns: # Somente se a coluna existir (já deveria pelo check acima)
                df_detailed[char_col] = pd.to_numeric(df_detailed[char_col], errors='coerce')
        
        df_detailed.dropna(subset=[metric_col_in_file, 'generation'] + char_cols_from_log, how='any', inplace=True) 
        if df_detailed.empty: return None
        return df_detailed[['generation', metric_col_in_file] + char_cols_from_log]
    except FileNotFoundError: return None
    except Exception as e:
        print(f"Erro em load_detailed_log_raw para L0={l0_size}, Métrica={metric_short_name}, Obj={goal_short_name}: {type(e).__name__} - {e}")
        return None
# -

In [27]:
# +
# (Código da Seção 9 como na sua última versão completa, para gerar df_summary_final_perf)
# ...
summary_table_data = []
if not AVAILABLE_L0_SIZES:
    print("Nenhum L0_SIZE encontrado. Pulando Seção 9.")
else:
    for l0_s_sum in AVAILABLE_L0_SIZES: 
        row_data = {"L0 Size": l0_s_sum}
        
        # Acurácia
        df_acc_min_sum = load_and_aggregate_detailed_log(l0_s_sum, "ACCURACY", "MINIMIZE", max_gens=MAX_GENERATIONS_TO_PLOT)
        df_acc_max_sum = load_and_aggregate_detailed_log(l0_s_sum, "ACCURACY", "MAXIMIZE", max_gens=MAX_GENERATIONS_TO_PLOT)
        acc_min_val = df_acc_min_sum['min_metric'].iloc[-1] if df_acc_min_sum is not None and not df_acc_min_sum.empty else np.nan
        acc_max_val = df_acc_max_sum['max_metric'].iloc[-1] if df_acc_max_sum is not None and not df_acc_max_sum.empty else np.nan
        row_data["Acurácia (AG Min)"] = f"{acc_min_val*100:.2f}%" if pd.notna(acc_min_val) else "N/A"
        row_data["Acurácia (AG Max)"] = f"{acc_max_val*100:.2f}%" if pd.notna(acc_max_val) else "N/A"
        
        # F1-Score
        df_f1_min_sum = load_and_aggregate_detailed_log(l0_s_sum, "F1", "MINIMIZE", max_gens=MAX_GENERATIONS_TO_PLOT)
        df_f1_max_sum = load_and_aggregate_detailed_log(l0_s_sum, "F1", "MAXIMIZE", max_gens=MAX_GENERATIONS_TO_PLOT)
        f1_min_val = df_f1_min_sum['min_metric'].iloc[-1] if df_f1_min_sum is not None and not df_f1_min_sum.empty else np.nan
        f1_max_val = df_f1_max_sum['max_metric'].iloc[-1] if df_f1_max_sum is not None and not df_f1_max_sum.empty else np.nan
        row_data["F1-Score (AG Min)"] = f"{f1_min_val*100:.2f}%" if pd.notna(f1_min_val) else "N/A"
        row_data["F1-Score (AG Max)"] = f"{f1_max_val*100:.2f}%" if pd.notna(f1_max_val) else "N/A"

        # Nº Gerações Exibidas
        all_gens_for_l0_sum = []
        if df_acc_min_sum is not None and not df_acc_min_sum.empty: all_gens_for_l0_sum.append(df_acc_min_sum['generation'].max())
        if df_acc_max_sum is not None and not df_acc_max_sum.empty: all_gens_for_l0_sum.append(df_acc_max_sum['generation'].max())
        if df_f1_min_sum is not None and not df_f1_min_sum.empty: all_gens_for_l0_sum.append(df_f1_min_sum['generation'].max())
        if df_f1_max_sum is not None and not df_f1_max_sum.empty: all_gens_for_l0_sum.append(df_f1_max_sum['generation'].max())
        max_gen_l0_sum = max(all_gens_for_l0_sum) if all_gens_for_l0_sum else 0
        row_data["Nº Gerações Exibidas"] = int(max_gen_l0_sum) if max_gen_l0_sum > 0 else "N/A"
        
        summary_table_data.append(row_data)

    if summary_table_data:
        df_summary_final_perf = pd.DataFrame(summary_table_data) # Renomeado para evitar conflito com df_summary_final da Seção 14 (delta)
        print("\n--- Tabela Resumo: Performance Final por Cenário (Última Geração Exibida) ---")
        try:
            from IPython.display import display, HTML
            display(HTML(df_summary_final_perf.to_html(index=False, classes='table table-striped')))
        except ImportError: print(df_summary_final_perf.to_string(index=False))
# -


--- Tabela Resumo: Performance Final por Cenário (Última Geração Exibida) ---


L0 Size,Acurácia (AG Min),Acurácia (AG Max),F1-Score (AG Min),F1-Score (AG Max),Nº Gerações Exibidas
10,0.09%,18.82%,0.01%,1.02%,100
50,7.13%,33.83%,0.57%,3.94%,100
100,10.86%,36.71%,1.19%,5.39%,100
500,36.17%,56.00%,6.52%,18.01%,100
1000,49.33%,62.20%,12.60%,24.69%,100
2500,63.92%,69.28%,24.11%,32.58%,100
5000,70.82%,74.13%,31.87%,40.07%,100
10000,75.10%,79.82%,38.44%,53.50%,100
20000,79.55%,83.23%,47.73%,61.53%,100
30000,83.03%,85.88%,54.05%,67.10%,100


In [29]:
# +
# (Código da Seção 15 (Heatmaps Apenas com Cores) da sua última resposta, que usa df_corr_summary, COLADO AQUI)
# ...
P_VALUE_SIGNIFICANCE_HEATMAP = 0.05 
heatmap_cmap = plt.cm.get_cmap('coolwarm') 
color_not_significant_bg = '#f0f0f0'; color_nan_bg = '#ffffff' 

def get_correlation_bgcolor(r, p_value): # ... (função como antes)
    if pd.isna(r): return f'background-color: {color_nan_bg}; color: #aaaaaa;'
    if pd.notna(p_value) and p_value >= P_VALUE_SIGNIFICANCE_HEATMAP: return f'background-color: {color_not_significant_bg}; color: dimgray;'
    norm_r_for_cmap = (r + 1) / 2
    rgba_color = heatmap_cmap(norm_r_for_cmap)
    hex_color = plt.colors.to_hex(rgba_color)
    luminance = 0.299*rgba_color[0] + 0.587*rgba_color[1] + 0.114*rgba_color[2]
    text_color = 'black' if luminance > 0.5 else 'white'
    return f'background-color: {hex_color}; color: {text_color};'

if 'df_corr_summary' not in locals() or df_corr_summary is None or df_corr_summary.empty:
    print("AVISO: df_corr_summary não disponível. Pulando Seção 11 (Heatmaps).")
else:
    char_display_map_heatmap = {'num_tokens': 'Nº Total de Tokens', 'num_distinct_tokens': 'Nº Tokens Distintos', 'num_classes_in_l0': 'Nº Classes Únicas no L0'}
    df_corr_summary_local_heatmap = df_corr_summary.copy()
    df_corr_summary_local_heatmap['r_p_tuple'] = list(zip(df_corr_summary_local_heatmap['Correlação Pearson (r)'], df_corr_summary_local_heatmap['p-valor (correlação)']))

    for char_col_heatmap, char_display_heatmap in char_display_map_heatmap.items():
        print(f"\n\n--- Heatmap de Correlação para: {char_display_heatmap} ---")
        df_char_heatmap_data = df_corr_summary_local_heatmap[df_corr_summary_local_heatmap['Característica'] == char_col_heatmap]
        if df_char_heatmap_data.empty: print(f"  Sem dados para {char_display_heatmap}."); continue
        
        try:
            pivot_table_r_values = df_char_heatmap_data.pivot_table(index="L0 Size", columns=["Métrica Otimizada", "Objetivo"], values="Correlação Pearson (r)", aggfunc='first')
            pivot_table_style_info = df_char_heatmap_data.pivot_table(index="L0 Size", columns=["Métrica Otimizada", "Objetivo"], values="r_p_tuple", aggfunc='first')
        except Exception as e: print(f"Erro ao pivotar para heatmap de {char_display_heatmap}: {e}"); continue
        if pivot_table_r_values.empty: print(f"  Tabela pivotada (r) vazia para {char_display_heatmap}."); continue

        # Função para aplicar o estilo célula a célula, buscando a tupla (r,p) na pivot_table_style_info
        def _apply_cell_style_from_tuple_lookup(r_val_in_cell, row_idx, col_multi_idx):
            # r_val_in_cell é o valor da célula da pivot_table_r_values
            # row_idx é o L0 Size
            # col_multi_idx é a tupla (Métrica Otimizada, Objetivo)
            try:
                r_tuple, p_tuple = pivot_table_style_info.loc[row_idx, col_multi_idx] # Acessa a tupla (r,p)
                return get_correlation_bgcolor(r_tuple, p_tuple)
            except (KeyError, TypeError, IndexError): # Se não encontrar a tupla ou for NaN
                return get_correlation_bgcolor(np.nan, np.nan)

        # Criar uma função que pode ser usada com Styler.apply
        # Ela precisa retornar um DataFrame de estilos com o mesmo shape
        def apply_heatmap_styles_to_df(df_r_values, df_style_tuples):
            style_df = pd.DataFrame('', index=df_r_values.index, columns=df_r_values.columns)
            for r_idx in df_r_values.index:
                for c_idx in df_r_values.columns:
                    tuple_val = df_style_tuples.loc[r_idx, c_idx]
                    if isinstance(tuple_val, tuple) and len(tuple_val) == 2:
                        style_df.loc[r_idx, c_idx] = get_correlation_bgcolor(tuple_val[0], tuple_val[1])
                    else:
                        style_df.loc[r_idx, c_idx] = get_correlation_bgcolor(np.nan, np.nan)
            return style_df
        
        styled_heatmap = pivot_table_r_values.style.format("{:.2f}", na_rep="N/A")\
            .set_table_styles([
                {'selector': 'th', 'props': [('font-size', '8pt'), ('text-align', 'center'), ('padding', '3px')]},
                {'selector': 'td', 'props': [('font-size', '8pt'), ('text-align', 'center'), ('min-width', '70px'), ('padding', '3px')]}
            ]).set_caption(f"Heatmap Correlação (r): {char_display_heatmap} vs Performance")\
            .apply(apply_heatmap_styles_to_df, axis=None, df_style_tuples=pivot_table_style_info) # Passar df_style_tuples como kwarg

        try:
            from IPython.display import display, HTML; display(styled_heatmap)
        except ImportError: print(pivot_table_r_values.to_string()) 
# -

C:\Users\ghdaru\AppData\Local\Temp\ipykernel_25812\2193814196.py:5: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  heatmap_cmap = plt.cm.get_cmap('coolwarm')




--- Heatmap de Correlação para: Nº Total de Tokens ---


AttributeError: module 'matplotlib.pyplot' has no attribute 'colors'



--- Heatmap de Correlação para: Nº Tokens Distintos ---


AttributeError: module 'matplotlib.pyplot' has no attribute 'colors'



--- Heatmap de Correlação para: Nº Classes Únicas no L0 ---


AttributeError: module 'matplotlib.pyplot' has no attribute 'colors'

In [23]:
# -*- coding: utf-8 -*-
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: light
#       format_version: '1.5'
#       jupytext_version: 1.14.5
#   kernelspec:
#     display_name: Python 3 (ipykernel)
#     language: python
#     name: python3
# ---

# # Análise dos Resultados da Otimização de L0 via Algoritmos Genéticos

# ## 1. Configurações e Imports

# +
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import seaborn as sns
import os
import glob
from collections import Counter

# NOVAS IMPORTAÇÕES PARA CORRELAÇÃO E REGRESSÃO
from scipy.stats import pearsonr
import statsmodels.api as sm # Opcional para regressão mais detalhada

# Adicionar o diretório pai ao sys.path para importar o módulo da biblioteca
import sys
# Presumindo que o notebook está em 'examples', e a biblioteca em 'activetextclassification' no nível acima
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

try:
    from activetextclassification.visualization.ag_plots_evolution import plot_population_evolution_combined
    print("Função de plotagem 'plot_population_evolution_combined' importada.")
except ImportError as e:
    print(f"Erro ao importar 'plot_population_evolution_combined': {e}")
    print("Certifique-se que 'activetextclassification/visualization/ag_plots_evolution.py' existe e o PYTHONPATH está correto.")
    # Definir uma função placeholder para evitar que o resto do notebook quebre
    def plot_population_evolution_combined(*args, **kwargs):
        print("ERRO: plot_population_evolution_combined não pôde ser importada. Gráfico não gerado.")


# Configurações de Estilo para os Gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) 
plt.rcParams['axes.titlesize'] = 14 # Reduzido para subplots
plt.rcParams['axes.labelsize'] = 12 # Reduzido
plt.rcParams['xtick.labelsize'] = 10 # Reduzido
plt.rcParams['ytick.labelsize'] = 10 # Reduzido
plt.rcParams['legend.fontsize'] = 8  # Reduzido
plt.rcParams['figure.titlesize'] = 16 # Para suptitle

# ## 2. Definição de Parâmetros e Caminhos 
# (sem grandes alterações, apenas garantindo que CONVERGENCE_L0_SIZES_EXAMPLE seja usado corretamente)

# +
base_ag_results_path = "." 
l0_size_folders = glob.glob(os.path.join(base_ag_results_path, "ag_optimization_results_L0_*"))
AVAILABLE_L0_SIZES = sorted([int(os.path.basename(f).split('_')[-1]) for f in l0_size_folders if os.path.basename(f).split('_')[-1].isdigit()])
print(f"Tamanhos de L0 com pastas de resultados AG encontradas: {AVAILABLE_L0_SIZES}")

L0_SIZES_PRIMARY = [10, 50, 100, 500, 1000, 2500, 5000, 10000, 20000, 30000, 100000]
L0_SIZES_FOR_PLOTS = [s for s in L0_SIZES_PRIMARY if s in AVAILABLE_L0_SIZES]
L0_SIZES_FOR_CURVA_OTIMA = [s for s in L0_SIZES_PRIMARY if s in AVAILABLE_L0_SIZES and s <= 30000]
if not L0_SIZES_FOR_PLOTS and AVAILABLE_L0_SIZES: 
    L0_SIZES_FOR_PLOTS = AVAILABLE_L0_SIZES
elif not L0_SIZES_FOR_PLOTS and not AVAILABLE_L0_SIZES:
    print("ALERTA: Nenhum L0_SIZE disponível para os gráficos de 'Curva Ótima' e 'Características'.")
    L0_SIZES_FOR_PLOTS = [10] 

# Para as tabelas de correlação, é bom ter uma lista abrangente de L0s com dados
L0_FOR_CORR_TABLES = [s for s in L0_SIZES_PRIMARY if s in AVAILABLE_L0_SIZES]
if not L0_FOR_CORR_TABLES: L0_FOR_CORR_TABLES = AVAILABLE_L0_SIZES # Fallback

print(f"Tamanhos de L0 que serão considerados para as tabelas de correlação: {L0_FOR_CORR_TABLES}")

# Para os gráficos de evolução da população, usaremos os exemplos da orientação
# ou o que estiver disponível.
CONVERGENCE_L0_SIZES_EXAMPLE = [s for s in [10, 50, 100, 500, 1000, 2500, 5000, 10000, 20000, 30000] if s in AVAILABLE_L0_SIZES]
if not CONVERGENCE_L0_SIZES_EXAMPLE and AVAILABLE_L0_SIZES: # Se os da orientação não existem, pega os primeiros disponíveis
    CONVERGENCE_L0_SIZES_EXAMPLE = AVAILABLE_L0_SIZES[:min(4, len(AVAILABLE_L0_SIZES))]
elif not CONVERGENCE_L0_SIZES_EXAMPLE and not AVAILABLE_L0_SIZES: # Se nada disponível
     print("ALERTA: Nenhum L0_SIZE disponível para os gráficos de 'Evolução da População'.")
     CONVERGENCE_L0_SIZES_EXAMPLE = []


print(f"Tamanhos de L0 para gráficos de 'Curva Ótima' e 'Características': {L0_SIZES_FOR_PLOTS}")
print(f"Tamanhos de L0 para gráficos de 'Evolução da População': {CONVERGENCE_L0_SIZES_EXAMPLE}")

RANDOM_STATS_FILE = os.path.join("data", "sensibilidade", "estatísticas.csv") 
FULL_DATASET_FILE = os.path.join("..", "data", "dataset.csv") 
TEXT_COLUMN = 'nm_item'; LABEL_COLUMN = 'nm_product'
AG_BEST_L0_BASE_NAME = "ag_best_l0"; AG_DETAILED_FITNESS_BASE_NAME = "ag_detailed_fitness" 
METRIC_MAP = {"ACCURACY": "Acurácia", "F1": "Macro F1-Score"}
GOAL_MAP = {"MAXIMIZE": "Maximização", "MINIMIZE": "Minimização"}

MAX_GENERATIONS_TO_PLOT = 100 # Limite de gerações para os plots
L0_SIZE_LIMIT_CURVA_OTIMA = 30000
MARKER_SIZE_CURVA_OTIMA = 4 # Novo parâmetro para tamanho dos marcadores na Seção 6
# -
# -

# ## 3. Carregamento de Dados 
# (Mantido como na versão anterior)

# ### 3.1 Dados da Amostragem Aleatória (do `estatisticas.csv`)

# +
df_random_detailed_stats = None
try:
    df_random_detailed_stats = pd.read_csv(RANDOM_STATS_FILE)
    print(f"Dados detalhados de amostragem aleatória carregados de: {RANDOM_STATS_FILE}")
    # Renomear colunas para consistência, se necessário (parecem ok pela amostra)
    # Converter l0_size para int se não for
    if 'l0_size' in df_random_detailed_stats.columns:
        df_random_detailed_stats['l0_size'] = pd.to_numeric(df_random_detailed_stats['l0_size'], errors='coerce')
        df_random_detailed_stats.dropna(subset=['l0_size'], inplace=True)
    else:
        print(f"AVISO: Coluna 'l0_size' não encontrada em {RANDOM_STATS_FILE}")
except FileNotFoundError: 
    print(f"ERRO: Arquivo de estatísticas aleatórias '{RANDOM_STATS_FILE}' não encontrado.")
except Exception as e: 
    print(f"ERRO ao carregar dados da amostragem aleatória de '{RANDOM_STATS_FILE}': {e}")

if df_random_detailed_stats is not None:
    display(df_random_detailed_stats.head())
# -

# ### 3.2 Dataset Completo (para análise de características dos L0s)

# +
df_full = None
try:
    df_full = pd.read_csv(FULL_DATASET_FILE)
    df_full.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN], inplace=True)
except FileNotFoundError: print(f"ERRO: Dataset completo não encontrado: {FULL_DATASET_FILE}"); df_full = None 
except Exception as e: print(f"ERRO ao carregar o dataset completo: {e}"); df_full = None
# -

# ## 4. Funções Auxiliares
# (Função load_and_aggregate_detailed_log CORRIGIDA)

# +
def get_ag_best_l0_filepath(l0_size, metric_short, goal_short):
    dir_path = os.path.join(base_ag_results_path, f"ag_optimization_results_L0_{l0_size}")
    filename = f"{AG_BEST_L0_BASE_NAME}_{metric_short.upper()}_{goal_short.upper()}.csv"
    return os.path.join(dir_path, filename)

def get_ag_detailed_fitness_filepath(l0_size, metric_short, goal_short):
    dir_path = os.path.join(base_ag_results_path, f"ag_optimization_results_L0_{l0_size}")
    filename = f"{AG_DETAILED_FITNESS_BASE_NAME}{metric_short.upper()}_{goal_short.upper()}.csv"
    return os.path.join(dir_path, filename)

# Função load_and_aggregate_detailed_log CORRIGIDA (essencial)
def load_and_aggregate_detailed_log(l0_size, metric_short_name, goal_short_name, max_gens=None):
    filepath = get_ag_detailed_fitness_filepath(l0_size, metric_short_name, goal_short_name)
    metric_col_in_file = 'accuracy_on_full' if metric_short_name.upper() == 'ACCURACY' else 'f1_macro_on_full'
    try:
        df_detailed = pd.read_csv(filepath)
        if max_gens and 'generation' in df_detailed.columns:
            df_detailed = df_detailed[df_detailed['generation'] <= max_gens]
        if df_detailed.empty: return None
        if metric_col_in_file not in df_detailed.columns: return None
        df_detailed[metric_col_in_file] = pd.to_numeric(df_detailed[metric_col_in_file], errors='coerce')
        char_cols = ['num_tokens', 'num_distinct_tokens', 'num_classes_in_l0']
        for char_col in char_cols:
            if char_col in df_detailed.columns:
                df_detailed[char_col] = pd.to_numeric(df_detailed[char_col], errors='coerce')
            else: df_detailed[char_col] = np.nan
        df_detailed.dropna(subset=[metric_col_in_file], inplace=True) 
        if df_detailed.empty: return None
        grouped = df_detailed.groupby('generation')
        df_agg_perf = grouped[metric_col_in_file].agg(['max', 'mean', 'min']).reset_index()
        df_agg_perf.rename(columns={'max': 'max_metric', 'mean': 'avg_metric', 'min': 'min_metric'}, inplace=True)
        idx_best_worst = grouped[metric_col_in_file].idxmax() if goal_short_name.upper() == 'MAXIMIZE' else grouped[metric_col_in_file].idxmin()
        df_best_worst_chars_raw = df_detailed.loc[idx_best_worst]
        existing_char_cols_for_merge = ['generation'] + [col for col in char_cols if col in df_best_worst_chars_raw.columns]
        df_best_worst_chars = df_best_worst_chars_raw[existing_char_cols_for_merge].reset_index(drop=True)
        if df_best_worst_chars.empty or not any(col in char_cols for col in df_best_worst_chars.columns):
            df_agg_final = df_agg_perf.copy()
            for col in char_cols: df_agg_final[col] = np.nan
        else:
            df_agg_final = pd.merge(df_agg_perf, df_best_worst_chars, on='generation', how='left')
        for col in char_cols:
            if col not in df_agg_final.columns: df_agg_final[col] = np.nan
        return df_agg_final
    except FileNotFoundError: return None
    except Exception as e:
        print(f"Erro em load_and_aggregate_detailed_log para L0={l0_size}, Métrica={metric_short_name}, Obj={goal_short_name}: {type(e).__name__} - {e}")
        return None

def load_detailed_log_raw(l0_size, metric_short_name, goal_short_name, max_gens=None):
    filepath = get_ag_detailed_fitness_filepath(l0_size, metric_short_name, goal_short_name)
    metric_col_in_file = 'accuracy_on_full' if metric_short_name.upper() == 'ACCURACY' else 'f1_macro_on_full'
    char_cols_from_log = ['num_tokens', 'num_distinct_tokens', 'num_classes_in_l0']
    try:
        df_detailed = pd.read_csv(filepath)
        if max_gens and 'generation' in df_detailed.columns:
            df_detailed = df_detailed[df_detailed['generation'] <= max_gens]
        if df_detailed.empty: return None
        required_cols = [metric_col_in_file, 'generation'] + char_cols_from_log
        if not all(col in df_detailed.columns for col in required_cols): return None
        df_detailed[metric_col_in_file] = pd.to_numeric(df_detailed[metric_col_in_file], errors='coerce')
        for char_col in char_cols_from_log:
            if char_col in df_detailed.columns:
                df_detailed[char_col] = pd.to_numeric(df_detailed[char_col], errors='coerce')
            else: df_detailed[char_col] = np.nan # Adicionar coluna se não existir
        # Apenas dropna se a métrica de performance ou características essenciais para correlação forem NaN
        # Para a correlação, precisamos de pares válidos.
        df_detailed.dropna(subset=[metric_col_in_file] + char_cols_from_log + ['generation'], how='any', inplace=True) 
        if df_detailed.empty: return None
        return df_detailed[['generation', metric_col_in_file] + char_cols_from_log]
    except FileNotFoundError: return None
    except Exception as e:
        print(f"Erro em load_detailed_log_raw para L0={l0_size}, Métrica={metric_short_name}, Obj={goal_short_name}: {type(e).__name__} - {e}")
        return None
# -


# ## 9. (Renumerado) Cálculo de `df_corr_summary` (Correlação por Cenário Individual)
# (Esta é a antiga Seção "Novo Código" ou "Seção 11" que gerava df_corr_summary)

# +
correlation_results_individual_scenario = [] 
characteristic_cols = ['num_tokens', 'num_distinct_tokens', 'num_classes_in_l0'] 
performance_metrics_cols_dict = { 
    "ACCURACY": "accuracy_on_full",
    "F1": "f1_macro_on_full"
}
scenarios_for_individual_corr = [
    ("ACCURACY", "MAXIMIZE"), ("ACCURACY", "MINIMIZE"),
    ("F1", "MAXIMIZE"), ("F1", "MINIMIZE")
]

l0_sizes_for_individual_corr = L0_FOR_CORR_TABLES 

df_corr_summary = pd.DataFrame() 

if not l0_sizes_for_individual_corr:
    print("Nenhum L0_SIZE definido em L0_FOR_CORR_TABLES. Pulando cálculo de df_corr_summary.")
else:
    print(f"\n--- Calculando Correlações por Cenário Individual para L0s: {l0_sizes_for_individual_corr} (até {MAX_GENERATIONS_TO_PLOT} ger.) ---")
    for l0_size_corr_ind in l0_sizes_for_individual_corr:
        for metric_short_corr_ind, goal_short_corr_ind in scenarios_for_individual_corr:
            df_raw_ind = load_detailed_log_raw(l0_size_corr_ind, metric_short_corr_ind, goal_short_corr_ind, max_gens=MAX_GENERATIONS_TO_PLOT)
            
            if df_raw_ind is None or df_raw_ind.empty:
                # print(f"  L0={l0_size_corr_ind}, {metric_short_corr_ind}-{goal_short_corr_ind}: Sem dados RAW.")
                # Adicionar entradas NaN para manter a estrutura da tabela
                for char_col_corr_ind_nan in characteristic_cols:
                    correlation_results_individual_scenario.append({
                        "L0 Size": l0_size_corr_ind, "Métrica Otimizada": METRIC_MAP[metric_short_corr_ind],
                        "Objetivo": GOAL_MAP[goal_short_corr_ind], "Característica": char_col_corr_ind_nan,
                        "Correlação Pearson (r)": np.nan, "p-valor (correlação)": np.nan,
                        "R² (Regressão Linear)": np.nan, "p-valor (slope)": np.nan,
                        "N Observações": 0
                    })
                continue # Pula para o próximo cenário
            
            perf_col_name_ind = performance_metrics_cols_dict[metric_short_corr_ind]
            
            for char_col_corr_ind in characteristic_cols:
                # Verificar novamente se as colunas existem após o carregamento (load_detailed_log_raw já verifica)
                if char_col_corr_ind not in df_raw_ind.columns or perf_col_name_ind not in df_raw_ind.columns:
                     correlation_results_individual_scenario.append({
                        "L0 Size": l0_size_corr_ind, "Métrica Otimizada": METRIC_MAP[metric_short_corr_ind],
                        "Objetivo": GOAL_MAP[goal_short_corr_ind], "Característica": char_col_corr_ind,
                        "Correlação Pearson (r)": np.nan, "p-valor (correlação)": np.nan,
                        "R² (Regressão Linear)": np.nan, "p-valor (slope)": np.nan,
                        "N Observações": 0
                    })
                     continue

                df_pair_ind = df_raw_ind[[perf_col_name_ind, char_col_corr_ind]].dropna()
                r_ind, p_corr_ind, r_sq_ind, p_slope_ind = np.nan, np.nan, np.nan, np.nan
                n_obs = len(df_pair_ind)

                if n_obs >= 2: 
                    try:
                        if n_obs > 2: r_ind, p_corr_ind = pearsonr(df_pair_ind[perf_col_name_ind], df_pair_ind[char_col_corr_ind])
                        elif n_obs == 2: # r é calculável, p-valor não
                             r_ind_calc = np.corrcoef(df_pair_ind[perf_col_name_ind], df_pair_ind[char_col_corr_ind])[0,1]
                             if np.isclose(abs(r_ind_calc), 1.0): r_ind = r_ind_calc
                             else: r_ind = np.nan # Evitar r indefinido se não for perfeitamente correlacionado com N=2
                             p_corr_ind = np.nan
                    except (ValueError, IndexError): r_ind, p_corr_ind = np.nan, np.nan
                
                if n_obs >=3: # Para OLS
                    X_ind = df_pair_ind[char_col_corr_ind]
                    y_ind = df_pair_ind[perf_col_name_ind]
                    if X_ind.nunique() > 1: 
                        X_ind_sm = sm.add_constant(X_ind, has_constant='add')
                        try:
                            model_fit_ind = sm.OLS(y_ind, X_ind_sm).fit()
                            r_sq_ind = model_fit_ind.rsquared
                            if len(model_fit_ind.pvalues) > 1: p_slope_ind = model_fit_ind.pvalues.iloc[1]
                        except Exception: pass 
                        
                correlation_results_individual_scenario.append({
                    "L0 Size": l0_size_corr_ind, "Métrica Otimizada": METRIC_MAP[metric_short_corr_ind],
                    "Objetivo": GOAL_MAP[goal_short_corr_ind], "Característica": char_col_corr_ind,
                    "Correlação Pearson (r)": r_ind, "p-valor (correlação)": p_corr_ind,
                    "R² (Regressão Linear)": r_sq_ind, "p-valor (slope)": p_slope_ind,
                    "N Observações": n_obs
                })

    if correlation_results_individual_scenario:
        df_corr_summary = pd.DataFrame(correlation_results_individual_scenario)
        print("\n--- Tabela Resumo de Correlações Individuais por Cenário (df_corr_summary) Gerada ---")
    else:
        print("Nenhum resultado de correlação individual para df_corr_summary.")
        df_corr_summary = pd.DataFrame(columns=[ # Criar com colunas esperadas se vazio
            "L0 Size", "Métrica Otimizada", "Objetivo", "Característica",
            "Correlação Pearson (r)", "p-valor (correlação)",
            "R² (Regressão Linear)", "p-valor (slope)", "N Observações"
        ])
# -

# ## 10. (Renumerado) Tabelas de Interpretação de Correlação (Heatmaps Textuais - Usa `df_corr_summary`)
# (Esta era a Seção 15 na sua solicitação, agora 10, dependendo da Seção 9 acima)

# +
# (Código da Seção 15 (Heatmaps) da sua última resposta, que usa df_corr_summary, COLADO AQUI)
# Limiares e descrições para correlação
CORR_THRESHOLDS = {
    "Forte Positiva": (0.7, 1.01), "Moderada Positiva": (0.4, 0.7), "Fraca Positiva": (0.1, 0.4),
    "Insensível": (-0.1, 0.1), "Fraca Negativa": (-0.4, -0.1),
    "Moderada Negativa": (-0.7, -0.4), "Forte Negativa": (-1.01, -0.7),
}
P_VALUE_SIGNIFICANCE_HEATMAP = 0.05 # Renomeado para evitar conflito se P_VALUE_SIGNIFICANCE for usado em outro lugar

CORR_COLORS_BG = {
    "Forte Positiva":    '#d62728', "Moderada Positiva": '#ff7f0e', "Fraca Positiva":    '#ffbb78',
    "Insensível":        '#f0f0f0', "Fraca Negativa":    '#aec7e8', "Moderada Negativa": '#1f77b4',
    "Forte Negativa":    '#084594', "Não Significativa": '#e0e0e0' 
}
CORR_COLORS_TEXT = {
    "Forte Positiva":    'white', "Moderada Positiva": 'black', "Fraca Positiva":    'black',
    "Insensível":        'black', "Fraca Negativa":    'black', "Moderada Negativa": 'white',
    "Forte Negativa":    'white', "Não Significativa": 'dimgray'
}

def interpret_correlation_for_heatmap(r, p_value): # Renomeado para clareza
    if pd.isna(r): # p_value pode ser NaN se r for NaN ou N<3
        return "N/A", "N/A", "lightgrey", "black" 
    
    text_val_display = f"{r:.2f}" # O que será exibido na célula se não for a descrição

    if pd.notna(p_value) and p_value >= P_VALUE_SIGNIFICANCE_HEATMAP:
        desc = f"Insensível (p={p_value:.2g})"
        bg_color = CORR_COLORS_BG["Não Significativa"]; text_color = CORR_COLORS_TEXT["Não Significativa"]
        # Para heatmap só com cores, retornaremos apenas r e as cores
        return r, desc, bg_color, text_color # Modificado para retornar r original para o styler.format

    # Se significativo
    for desc_key, (low, high) in CORR_THRESHOLDS.items():
        if low <= r < high:
            # desc_with_r_val = f"{desc_key} (r={r:.2f})" # Não precisamos mais disso para display
            if desc_key == "Insensível": # Já é significativo, então r está próximo de zero
                 desc_for_display = f"Próx. Zero (r={r:.2f}, p={p_value:.2g})"
                 bg_color = CORR_COLORS_BG["Insensível"]; text_color = CORR_COLORS_TEXT["Insensível"]
            else:
                desc_for_display = f"{desc_key} (r={r:.2f})" # Manter este para referência, mas não será o display principal
                bg_color = CORR_COLORS_BG[desc_key]; text_color = CORR_COLORS_TEXT[desc_key]
            return r, desc_for_display, bg_color, text_color # Retorna r original, descrição completa, e cores
    return r, "Erro", "white", "black" # Fallback

def style_cell_for_heatmap(val_tuple_r_desc_colors): 
    if isinstance(val_tuple_r_desc_colors, tuple) and len(val_tuple_r_desc_colors) == 4:
        _, _, bg_color, text_color = val_tuple_r_desc_colors # Ignora r e desc para o estilo direto
        return f'background-color: {bg_color}; color: {text_color};'
    return '' # Estilo vazio se não for tupla


if 'df_corr_summary' not in locals() or df_corr_summary is None or df_corr_summary.empty:
    print("AVISO: df_corr_summary não está disponível ou está vazio. Pulando Seção 10 (Heatmaps).")
else:
    char_display_map_heatmap = { # Renomeado para evitar conflito
        'num_tokens': 'Nº Total de Tokens',
        'num_distinct_tokens': 'Nº Tokens Distintos',
        'num_classes_in_l0': 'Nº Classes Únicas no L0'
    }
    for char_col_table, char_display_table in char_display_map_heatmap.items():
        print(f"\n\n--- Heatmap de Correlação para: {char_display_table} ---")
        df_char_corr_heatmap_src = df_corr_summary[df_corr_summary['Característica'] == char_col_table].copy()
        if df_char_corr_heatmap_src.empty: print(f"  Sem dados para {char_display_table}."); continue

        # Coluna para aplicar o Styler (contém tuplas com r, p_val, e info de estilo)
        df_char_corr_heatmap_src['StyleInfo'] = df_char_corr_heatmap_src.apply(
            lambda row: interpret_correlation_for_heatmap(row['Correlação Pearson (r)'], row['p-valor (correlação)']), axis=1
        )
        
        try:
            # Tabela pivotada com os valores de 'r' para exibição
            pivot_table_display_r = df_char_corr_heatmap_src.pivot_table(
                index="L0 Size", columns=["Métrica Otimizada", "Objetivo"],
                values="Correlação Pearson (r)", aggfunc='first')

            # Tabela pivotada com as tuplas de 'StyleInfo' para aplicar estilos
            pivot_table_for_style = df_char_corr_heatmap_src.pivot_table(
                index="L0 Size", columns=["Métrica Otimizada", "Objetivo"],
                values="StyleInfo", aggfunc='first')
                
        except Exception as e: print(f"Erro ao pivotar para heatmap de {char_display_table}: {e}"); continue
            
        if pivot_table_display_r.empty: print(f"  Tabela pivotada (r) vazia para {char_display_table}."); continue

        # Função para aplicar o estilo dinamicamente
        def _apply_heatmap_styles(data_series_of_r_values): # data_series_of_r_values é uma coluna da pivot_table_display_r
            # O nome da série (data_series_of_r_values.name) é uma tupla (Métrica Otimizada, Objetivo)
            # O índice da série (data_series_of_r_values.index) é o L0 Size
            
            # Recuperar as tuplas de estilo da pivot_table_for_style
            # E aplicar estilo baseado nelas
            
            styles = []
            for l0_idx, r_val in data_series_of_r_values.items():
                # Acessar a tupla de estilo correspondente
                metric_col_level0, obj_col_level1 = data_series_of_r_values.name
                style_tuple = pivot_table_for_style.loc[l0_idx, (metric_col_level0, obj_col_level1)]
                styles.append(style_cell_for_heatmap(style_tuple))
            return styles

        styled_heatmap = pivot_table_display_r.style.format("{:.2f}", na_rep="N/A")\
            .apply(_apply_heatmap_styles, axis=0)\
            .set_table_styles([
                {'selector': 'th', 'props': [('font-size', '8pt'), ('text-align', 'center'), ('padding', '3px')]}, # Reduzido th
                {'selector': 'td', 'props': [('font-size', '8pt'), ('text-align', 'center'), ('min-width', '70px'), ('padding', '3px')]}  # Reduzido td
            ]).set_caption(f"Heatmap Correlação (r): {char_display_table} vs Performance (Corpo: r, Fundo: Força/Signif.)")

        try:
            from IPython.display import display, HTML; display(styled_heatmap)
        except ImportError: print(pivot_table_display_r.to_string())
# -


###############################################################  Versão 02
# +
def get_ag_detailed_fitness_filepath(l0_size, metric_short, goal_short):
    dir_path = os.path.join(base_ag_results_path, f"ag_optimization_results_L0_{l0_size}")
    filename = f"ag_detailed_fitness{metric_short.upper()}_{goal_short.upper()}.csv"
    return os.path.join(dir_path, filename)

def load_and_aggregate_detailed_log(l0_size, metric_short_name, goal_short_name, max_gens=None):
    filepath = get_ag_detailed_fitness_filepath(l0_size, metric_short_name, goal_short_name)
    metric_col_in_file = 'accuracy_on_full' if metric_short_name.upper() == 'ACCURACY' else 'f1_macro_on_full'
    try:
        df_detailed = pd.read_csv(filepath)
        if max_gens and 'generation' in df_detailed.columns:
            df_detailed = df_detailed[df_detailed['generation'] <= max_gens]
        if df_detailed.empty: return None
        if metric_col_in_file not in df_detailed.columns: return None
        df_detailed[metric_col_in_file] = pd.to_numeric(df_detailed[metric_col_in_file], errors='coerce')
        char_cols = ['num_tokens', 'num_distinct_tokens', 'num_classes_in_l0']
        for char_col in char_cols:
            if char_col in df_detailed.columns:
                df_detailed[char_col] = pd.to_numeric(df_detailed[char_col], errors='coerce')
            else: df_detailed[char_col] = np.nan
        df_detailed.dropna(subset=[metric_col_in_file], inplace=True) 
        if df_detailed.empty: return None
        grouped = df_detailed.groupby('generation')
        df_agg_perf = grouped[metric_col_in_file].agg(['max', 'mean', 'min']).reset_index()
        df_agg_perf.rename(columns={'max': 'max_metric', 'mean': 'avg_metric', 'min': 'min_metric'}, inplace=True)
        
        if goal_short_name.upper() == 'MAXIMIZE':
            idx_best_worst = grouped[metric_col_in_file].idxmax()
        else: 
            idx_best_worst = grouped[metric_col_in_file].idxmin()
        
        df_best_worst_chars_raw = df_detailed.loc[idx_best_worst]
        existing_char_cols_for_merge = ['generation'] + [col for col in char_cols if col in df_best_worst_chars_raw.columns]
        df_best_worst_chars = df_best_worst_chars_raw[existing_char_cols_for_merge].reset_index(drop=True)

        if df_best_worst_chars.empty or not any(col in char_cols for col in df_best_worst_chars.columns if col in df_best_worst_chars): # Check if any relevant char_col exists
            df_agg_final = df_agg_perf.copy()
            for col in char_cols: df_agg_final[col] = np.nan
        else:
            df_agg_final = pd.merge(df_agg_perf, df_best_worst_chars, on='generation', how='left')
        
        for col in char_cols: # Ensure all characteristic columns are present, even if all NaN
            if col not in df_agg_final.columns:
                df_agg_final[col] = np.nan
        return df_agg_final
    except FileNotFoundError: return None
    except Exception as e:
        print(f"Erro em load_and_aggregate_detailed_log para L0={l0_size}, Métrica={metric_short_name}, Obj={goal_short_name}: {type(e).__name__} - {e}")
        return None

def load_detailed_log_raw(l0_size, metric_short_name, goal_short_name, max_gens=None):
    filepath = get_ag_detailed_fitness_filepath(l0_size, metric_short_name, goal_short_name)
    metric_col_in_file = 'accuracy_on_full' if metric_short_name.upper() == 'ACCURACY' else 'f1_macro_on_full'
    char_cols_from_log = ['num_tokens', 'num_distinct_tokens', 'num_classes_in_l0']
    try:
        df_detailed = pd.read_csv(filepath)
        if max_gens and 'generation' in df_detailed.columns:
            df_detailed = df_detailed[df_detailed['generation'] <= max_gens]
        if df_detailed.empty: return None
        
        # Checagem mais robusta de colunas
        required_cols_check = [metric_col_in_file, 'generation'] + char_cols_from_log
        for col_check in required_cols_check:
            if col_check not in df_detailed.columns:
                # print(f"AVISO: Coluna RAW '{col_check}' ausente em {filepath} para L0={l0_size}.")
                # Adicionar coluna com NaNs se não existir, para evitar falhas posteriores
                if col_check not in df_detailed.columns: df_detailed[col_check] = np.nan

        df_detailed[metric_col_in_file] = pd.to_numeric(df_detailed[metric_col_in_file], errors='coerce')
        for char_col in char_cols_from_log:
             if char_col in df_detailed.columns: # Somente se a coluna existir (já deveria pelo check acima)
                df_detailed[char_col] = pd.to_numeric(df_detailed[char_col], errors='coerce')
        
        df_detailed.dropna(subset=[metric_col_in_file, 'generation'] + char_cols_from_log, how='any', inplace=True) 
        if df_detailed.empty: return None
        return df_detailed[['generation', metric_col_in_file] + char_cols_from_log]
    except FileNotFoundError: return None
    except Exception as e:
        print(f"Erro em load_detailed_log_raw para L0={l0_size}, Métrica={metric_short_name}, Obj={goal_short_name}: {type(e).__name__} - {e}")
        return None
# -
# ## 9. Resumo dos Resultados da Otimização (Performance Final por Cenário)
# (Esta seção usa `load_and_aggregate_detailed_log`)

# +
# (Código da Seção 9 como na sua última versão completa, para gerar df_summary_final_perf)
# ...
summary_table_data = []
if not AVAILABLE_L0_SIZES:
    print("Nenhum L0_SIZE encontrado. Pulando Seção 9.")
else:
    for l0_s_sum in AVAILABLE_L0_SIZES: 
        row_data = {"L0 Size": l0_s_sum}
        
        # Acurácia
        df_acc_min_sum = load_and_aggregate_detailed_log(l0_s_sum, "ACCURACY", "MINIMIZE", max_gens=MAX_GENERATIONS_TO_PLOT)
        df_acc_max_sum = load_and_aggregate_detailed_log(l0_s_sum, "ACCURACY", "MAXIMIZE", max_gens=MAX_GENERATIONS_TO_PLOT)
        acc_min_val = df_acc_min_sum['min_metric'].iloc[-1] if df_acc_min_sum is not None and not df_acc_min_sum.empty else np.nan
        acc_max_val = df_acc_max_sum['max_metric'].iloc[-1] if df_acc_max_sum is not None and not df_acc_max_sum.empty else np.nan
        row_data["Acurácia (AG Min)"] = f"{acc_min_val*100:.2f}%" if pd.notna(acc_min_val) else "N/A"
        row_data["Acurácia (AG Max)"] = f"{acc_max_val*100:.2f}%" if pd.notna(acc_max_val) else "N/A"
        
        # F1-Score
        df_f1_min_sum = load_and_aggregate_detailed_log(l0_s_sum, "F1", "MINIMIZE", max_gens=MAX_GENERATIONS_TO_PLOT)
        df_f1_max_sum = load_and_aggregate_detailed_log(l0_s_sum, "F1", "MAXIMIZE", max_gens=MAX_GENERATIONS_TO_PLOT)
        f1_min_val = df_f1_min_sum['min_metric'].iloc[-1] if df_f1_min_sum is not None and not df_f1_min_sum.empty else np.nan
        f1_max_val = df_f1_max_sum['max_metric'].iloc[-1] if df_f1_max_sum is not None and not df_f1_max_sum.empty else np.nan
        row_data["F1-Score (AG Min)"] = f"{f1_min_val*100:.2f}%" if pd.notna(f1_min_val) else "N/A"
        row_data["F1-Score (AG Max)"] = f"{f1_max_val*100:.2f}%" if pd.notna(f1_max_val) else "N/A"

        # Nº Gerações Exibidas
        all_gens_for_l0_sum = []
        if df_acc_min_sum is not None and not df_acc_min_sum.empty: all_gens_for_l0_sum.append(df_acc_min_sum['generation'].max())
        if df_acc_max_sum is not None and not df_acc_max_sum.empty: all_gens_for_l0_sum.append(df_acc_max_sum['generation'].max())
        if df_f1_min_sum is not None and not df_f1_min_sum.empty: all_gens_for_l0_sum.append(df_f1_min_sum['generation'].max())
        if df_f1_max_sum is not None and not df_f1_max_sum.empty: all_gens_for_l0_sum.append(df_f1_max_sum['generation'].max())
        max_gen_l0_sum = max(all_gens_for_l0_sum) if all_gens_for_l0_sum else 0
        row_data["Nº Gerações Exibidas"] = int(max_gen_l0_sum) if max_gen_l0_sum > 0 else "N/A"
        
        summary_table_data.append(row_data)

    if summary_table_data:
        df_summary_final_perf = pd.DataFrame(summary_table_data) # Renomeado para evitar conflito com df_summary_final da Seção 14 (delta)
        print("\n--- Tabela Resumo: Performance Final por Cenário (Última Geração Exibida) ---")
        try:
            from IPython.display import display, HTML
            display(HTML(df_summary_final_perf.to_html(index=False, classes='table table-striped')))
        except ImportError: print(df_summary_final_perf.to_string(index=False))
# -

# ## 10. Cálculo de `df_corr_summary` (Correlação por Cenário Individual)
# (Este é o código que gera `df_corr_summary`, que a Seção 11 - Heatmaps - precisa)

# +
# (Código da Seção 10 da sua última resposta completa, que gera df_corr_summary, COLADO AQUI)
# ...
correlation_results_individual_scenario = [] 
characteristic_cols = ['num_tokens', 'num_distinct_tokens', 'num_classes_in_l0'] 
performance_metrics_cols_dict = { "ACCURACY": "accuracy_on_full", "F1": "f1_macro_on_full" }
scenarios_for_individual_corr = [ ("ACCURACY", "MAXIMIZE"), ("ACCURACY", "MINIMIZE"), ("F1", "MAXIMIZE"), ("F1", "MINIMIZE")]
l0_sizes_for_individual_corr = L0_FOR_CORR_TABLES 
df_corr_summary = pd.DataFrame() 

if not l0_sizes_for_individual_corr:
    print("Nenhum L0_SIZE definido. Pulando cálculo de df_corr_summary.")
else:
    print(f"\n--- Calculando Correlações por Cenário Individual para L0s: {l0_sizes_for_individual_corr} (até {MAX_GENERATIONS_TO_PLOT} ger.) ---")
    # (Resto do loop e cálculos como na sua célula "Novo Código: Seção que Gera df_corr_summary")
    for l0_size_corr_ind in l0_sizes_for_individual_corr:
        for metric_short_corr_ind, goal_short_corr_ind in scenarios_for_individual_corr:
            df_raw_ind = load_detailed_log_raw(l0_size_corr_ind, metric_short_corr_ind, goal_short_corr_ind, max_gens=MAX_GENERATIONS_TO_PLOT)
            if df_raw_ind is None or df_raw_ind.empty:
                for char_col_corr_ind_nan in characteristic_cols:
                    correlation_results_individual_scenario.append({"L0 Size": l0_size_corr_ind, "Métrica Otimizada": METRIC_MAP[metric_short_corr_ind],"Objetivo": GOAL_MAP[goal_short_corr_ind], "Característica": char_col_corr_ind_nan,"Correlação Pearson (r)": np.nan, "p-valor (correlação)": np.nan,"R² (Regressão Linear)": np.nan, "p-valor (slope)": np.nan,"N Observações": 0})
                continue
            perf_col_name_ind = performance_metrics_cols_dict[metric_short_corr_ind]
            for char_col_corr_ind in characteristic_cols:
                if char_col_corr_ind not in df_raw_ind.columns or perf_col_name_ind not in df_raw_ind.columns or df_raw_ind[char_col_corr_ind].isnull().all() or df_raw_ind[perf_col_name_ind].isnull().all():
                     correlation_results_individual_scenario.append({"L0 Size": l0_size_corr_ind, "Métrica Otimizada": METRIC_MAP[metric_short_corr_ind],"Objetivo": GOAL_MAP[goal_short_corr_ind], "Característica": char_col_corr_ind,"Correlação Pearson (r)": np.nan, "p-valor (correlação)": np.nan,"R² (Regressão Linear)": np.nan, "p-valor (slope)": np.nan,"N Observações": 0})
                     continue
                df_pair_ind = df_raw_ind[[perf_col_name_ind, char_col_corr_ind]].dropna()
                r_ind, p_corr_ind, r_sq_ind, p_slope_ind = np.nan, np.nan, np.nan, np.nan
                n_obs = len(df_pair_ind)
                if n_obs >= 2: 
                    try:
                        if n_obs > 2: r_ind, p_corr_ind = pearsonr(df_pair_ind[perf_col_name_ind], df_pair_ind[char_col_corr_ind])
                        elif n_obs == 2: 
                             r_ind_calc = np.corrcoef(df_pair_ind[perf_col_name_ind], df_pair_ind[char_col_corr_ind])[0,1]
                             if np.isclose(abs(r_ind_calc), 1.0): r_ind = r_ind_calc
                             else: r_ind = np.nan 
                             p_corr_ind = np.nan
                    except (ValueError, IndexError): r_ind, p_corr_ind = np.nan, np.nan
                if n_obs >=3: 
                    X_ind = df_pair_ind[char_col_corr_ind]
                    y_ind = df_pair_ind[perf_col_name_ind]
                    if X_ind.nunique() > 1: 
                        X_ind_sm = sm.add_constant(X_ind, has_constant='add')
                        try:
                            model_fit_ind = sm.OLS(y_ind, X_ind_sm).fit()
                            r_sq_ind = model_fit_ind.rsquared
                            if len(model_fit_ind.pvalues) > 1: p_slope_ind = model_fit_ind.pvalues.iloc[1]
                        except Exception: pass                         
                correlation_results_individual_scenario.append({"L0 Size": l0_size_corr_ind, "Métrica Otimizada": METRIC_MAP[metric_short_corr_ind],"Objetivo": GOAL_MAP[goal_short_corr_ind], "Característica": char_col_corr_ind,"Correlação Pearson (r)": r_ind, "p-valor (correlação)": p_corr_ind,"R² (Regressão Linear)": r_sq_ind, "p-valor (slope)": p_slope_ind,"N Observações": n_obs})
    if correlation_results_individual_scenario:
        df_corr_summary = pd.DataFrame(correlation_results_individual_scenario)
        print("\n--- Tabela Resumo de Correlações Individuais por Cenário (df_corr_summary) Gerada ---")
    else:
        print("Nenhum resultado de correlação individual para df_corr_summary.")
        df_corr_summary = pd.DataFrame(columns=["L0 Size", "Métrica Otimizada", "Objetivo", "Característica","Correlação Pearson (r)", "p-valor (correlação)","R² (Regressão Linear)", "p-valor (slope)", "N Observações"])
# -

# ## 11. Tabelas de Interpretação de Correlação (Heatmaps - Apenas Cores)
# (Esta era a Seção 15 na sua solicitação, agora 11, dependendo da Seção 10 acima)

# +
# (Código da Seção 15 (Heatmaps Apenas com Cores) da sua última resposta, que usa df_corr_summary, COLADO AQUI)
# ...
P_VALUE_SIGNIFICANCE_HEATMAP = 0.05 
heatmap_cmap = plt.cm.get_cmap('coolwarm') 
color_not_significant_bg = '#f0f0f0'; color_nan_bg = '#ffffff' 

def get_correlation_bgcolor(r, p_value): # ... (função como antes)
    if pd.isna(r): return f'background-color: {color_nan_bg}; color: #aaaaaa;'
    if pd.notna(p_value) and p_value >= P_VALUE_SIGNIFICANCE_HEATMAP: return f'background-color: {color_not_significant_bg}; color: dimgray;'
    norm_r_for_cmap = (r + 1) / 2
    rgba_color = heatmap_cmap(norm_r_for_cmap)
    hex_color = plt.colors.to_hex(rgba_color)
    luminance = 0.299*rgba_color[0] + 0.587*rgba_color[1] + 0.114*rgba_color[2]
    text_color = 'black' if luminance > 0.5 else 'white'
    return f'background-color: {hex_color}; color: {text_color};'

if 'df_corr_summary' not in locals() or df_corr_summary is None or df_corr_summary.empty:
    print("AVISO: df_corr_summary não disponível. Pulando Seção 11 (Heatmaps).")
else:
    char_display_map_heatmap = {'num_tokens': 'Nº Total de Tokens', 'num_distinct_tokens': 'Nº Tokens Distintos', 'num_classes_in_l0': 'Nº Classes Únicas no L0'}
    df_corr_summary_local_heatmap = df_corr_summary.copy()
    df_corr_summary_local_heatmap['r_p_tuple'] = list(zip(df_corr_summary_local_heatmap['Correlação Pearson (r)'], df_corr_summary_local_heatmap['p-valor (correlação)']))

    for char_col_heatmap, char_display_heatmap in char_display_map_heatmap.items():
        print(f"\n\n--- Heatmap de Correlação para: {char_display_heatmap} ---")
        df_char_heatmap_data = df_corr_summary_local_heatmap[df_corr_summary_local_heatmap['Característica'] == char_col_heatmap]
        if df_char_heatmap_data.empty: print(f"  Sem dados para {char_display_heatmap}."); continue
        
        try:
            pivot_table_r_values = df_char_heatmap_data.pivot_table(index="L0 Size", columns=["Métrica Otimizada", "Objetivo"], values="Correlação Pearson (r)", aggfunc='first')
            pivot_table_style_info = df_char_heatmap_data.pivot_table(index="L0 Size", columns=["Métrica Otimizada", "Objetivo"], values="r_p_tuple", aggfunc='first')
        except Exception as e: print(f"Erro ao pivotar para heatmap de {char_display_heatmap}: {e}"); continue
        if pivot_table_r_values.empty: print(f"  Tabela pivotada (r) vazia para {char_display_heatmap}."); continue

        # Função para aplicar o estilo célula a célula, buscando a tupla (r,p) na pivot_table_style_info
        def _apply_cell_style_from_tuple_lookup(r_val_in_cell, row_idx, col_multi_idx):
            # r_val_in_cell é o valor da célula da pivot_table_r_values
            # row_idx é o L0 Size
            # col_multi_idx é a tupla (Métrica Otimizada, Objetivo)
            try:
                r_tuple, p_tuple = pivot_table_style_info.loc[row_idx, col_multi_idx] # Acessa a tupla (r,p)
                return get_correlation_bgcolor(r_tuple, p_tuple)
            except (KeyError, TypeError, IndexError): # Se não encontrar a tupla ou for NaN
                return get_correlation_bgcolor(np.nan, np.nan)

        # Criar uma função que pode ser usada com Styler.apply
        # Ela precisa retornar um DataFrame de estilos com o mesmo shape
        def apply_heatmap_styles_to_df(df_r_values, df_style_tuples):
            style_df = pd.DataFrame('', index=df_r_values.index, columns=df_r_values.columns)
            for r_idx in df_r_values.index:
                for c_idx in df_r_values.columns:
                    tuple_val = df_style_tuples.loc[r_idx, c_idx]
                    if isinstance(tuple_val, tuple) and len(tuple_val) == 2:
                        style_df.loc[r_idx, c_idx] = get_correlation_bgcolor(tuple_val[0], tuple_val[1])
                    else:
                        style_df.loc[r_idx, c_idx] = get_correlation_bgcolor(np.nan, np.nan)
            return style_df
        
        styled_heatmap = pivot_table_r_values.style.format("{:.2f}", na_rep="N/A")\
            .set_table_styles([
                {'selector': 'th', 'props': [('font-size', '8pt'), ('text-align', 'center'), ('padding', '3px')]},
                {'selector': 'td', 'props': [('font-size', '8pt'), ('text-align', 'center'), ('min-width', '70px'), ('padding', '3px')]}
            ]).set_caption(f"Heatmap Correlação (r): {char_display_heatmap} vs Performance")\
            .apply(apply_heatmap_styles_to_df, axis=None, df_style_tuples=pivot_table_style_info) # Passar df_style_tuples como kwarg

        try:
            from IPython.display import display, HTML; display(styled_heatmap)
        except ImportError: print(pivot_table_r_values.to_string()) 
# -




Função de plotagem 'plot_population_evolution_combined' importada.
Tamanhos de L0 com pastas de resultados AG encontradas: [10, 50, 100, 500, 1000, 2500, 5000, 10000, 20000, 30000]
Tamanhos de L0 que serão considerados para as tabelas de correlação: [10, 50, 100, 500, 1000, 2500, 5000, 10000, 20000, 30000]
Tamanhos de L0 para gráficos de 'Curva Ótima' e 'Características': [10, 50, 100, 500, 1000, 2500, 5000, 10000, 20000, 30000]
Tamanhos de L0 para gráficos de 'Evolução da População': [10, 50, 100, 500, 1000, 2500, 5000, 10000, 20000, 30000]
Dados detalhados de amostragem aleatória carregados de: data\sensibilidade\estatísticas.csv


,Métrica,l0_size,Média,Mediana,DesvioPadrão,Mínimo,P25,P75,Máximo,IQR,CV (Mediana)
0,Acurácia,10,0.066649,0.068869,0.023793,0.025603,0.045918,0.088246,0.102171,0.042328,0.345485
1,Acurácia,20,0.105012,0.110059,0.023253,0.049388,0.089185,0.121807,0.138118,0.032622,0.211274
2,Acurácia,30,0.139531,0.138678,0.022874,0.095281,0.125312,0.155383,0.181395,0.030071,0.164945
3,Acurácia,40,0.159713,0.159447,0.022555,0.100274,0.144589,0.178245,0.191021,0.033656,0.141460
4,Acurácia,50,0.178486,0.179758,0.021830,0.124079,0.164435,0.198550,0.208496,0.034115,0.121440



--- Calculando Correlações por Cenário Individual para L0s: [10, 50, 100, 500, 1000, 2500, 5000, 10000, 20000, 30000] (até 100 ger.) ---

--- Tabela Resumo de Correlações Individuais por Cenário (df_corr_summary) Gerada ---


--- Heatmap de Correlação para: Nº Total de Tokens ---




--- Heatmap de Correlação para: Nº Tokens Distintos ---




--- Heatmap de Correlação para: Nº Classes Únicas no L0 ---


...

AttributeError: module 'matplotlib.pyplot' has no attribute 'colors'



--- Heatmap de Correlação para: Nº Tokens Distintos ---


AttributeError: module 'matplotlib.pyplot' has no attribute 'colors'



--- Heatmap de Correlação para: Nº Classes Únicas no L0 ---


AttributeError: module 'matplotlib.pyplot' has no attribute 'colors'